In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [ ]:
# 문서 출력 도우미 함수
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
documents = TextLoader("./data/appendix-keywords.txt").load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/msmarco-distilbert-dot-v5")

In [ ]:
retriever = FAISS.from_documents(texts, embeddings).as_retriver(search_kwargs={"k": 10})

In [ ]:
query = "Word2Vec 에 대해서 알려줄래?"

In [ ]:
docs = retriever.invoke(query)  # 질의를 수행하고 결과 문서 반환하기

In [ ]:
pretty_print_docs(docs)  # 결과 문서 출력

Reranker 사용하기

In [ ]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

In [ ]:
compressor = CrossEncoderReranker(model=model, top_n=3)

compression_retriever = ContextialCompresionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [ ]:
compressed_docs = compression_retriever.invoke("Word2Vec 에 대해서 알려줄래?")

In [ ]:
pretty_print_docs(compressed_docs)

In [ ]:
compressed_docs